In [1]:
from dotenv import load_dotenv
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv('../../env', override=True)
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME="gpt-4.1-mini",
print(AZURE_OPENAI_API_KEY[:10])
print(MODEL_NAME)

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('EMB_END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv('LANGCHAIN_ENDPOINT')
os.environ['LANGCHAIN_TRACING_V2'] = 'false' #true, false
os.environ['LANGCHAIN_PROJECT'] = 'RAG'

if os.getenv('LANGCHAIN_TRACING_V2') == "true":
    if len(os.getenv('LANGCHAIN_API_KEY')) > 0:
        print('랭스미스로 추적 중입니다 :', os.getenv('LANGSMITH_API_KEY')[:10])
    else:
        print('랭스미스 키가 확인되지 않았습니다.')

3BHTZdVIXx
('gpt-4.1-mini',)
랭스미스로 추적 중입니다 : lsv2_pt_ef


In [2]:
from langchain_openai import AzureChatOpenAI
llm = AzureChatOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=END_POINT,
    azure_deployment="gpt-4.1-mini",
    temperature=0,
    api_version="2024-12-01-preview",
)

from langchain_openai import AzureOpenAIEmbeddings
emb = AzureOpenAIEmbeddings(
    model=EMB_MODEL_NAME,
    api_key=AZURE_OPENAI_EMB_API_KEY,
    azure_endpoint=EMB_END_POINT,
    api_version="2024-08-01-preview"
)

In [3]:
!pip install sacrebleu rouge_score nest_asyncio
import nest_asyncio
nest_asyncio.apply()

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

  Installing build dependencies ... - \ done
  Getting requirements to build wheel ... - done
  Preparing metadata (pyproject.toml) ... - done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 49.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 48.4 MB/s  0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24987 sha256=2d5c9772cd8b9840344d79f76a8819b6929e1c9ec2fdfceb5e717b0d04c7166a
  Stored in directory: /home/azureuser/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [rouge_score] [lxml]


# RAGAS를 이용한 llm 어플리케이션 자동 평가

https://docs.ragas.io/en/latest/

https://docs.ragas.io/en/latest/references/metrics/#ragas.metrics

In [23]:
%pip install sacrebleu rouge_score

  Using cached rouge_score-0.1.2.tar.gz (17 kB)
  Installing build dependencies ... - \ done
  Getting requirements to build wheel ... - done
  Preparing metadata (pyproject.toml) ... - done
  Using cached nltk-3.9.3-py3-none-any.whl.metadata (3.2 kB)
Using cached nltk-3.9.3-py3-none-any.whl (1.5 MB)
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24987 sha256=758a2c59e282ee9e5cbebf5ebd280bdd273feea5892d2ef87f4a6b7efdc9b997
  Stored in directory: /home/azureuser/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [rouge_score] [nltk]
Note: you may need to restart the kernel to use updated packages.


In [24]:
from ragas import SingleTurnSample
from ragas.metrics import BleuScore, RougeScore

test_data = {
    "user_input": "다음 글을 요약하세요.\n회사는 2024년 3분기에 8%의 상승을 기록했으며, 이는 아시아 시장에서의 강력한 실적에 힘입은 것입니다. 이 지역의 판매가 전체 성장에 크게 기여했습니다. 분석가들은 이러한 성공을 전략적인 마케팅과 제품 현지화 덕분으로 보고 있습니다. 아시아 시장의 긍정적인 추세는 다음 분기에도 이어질 것으로 예상됩니다.",
    "response": "회사는 2024년 3분기에 8%의 성장을 기록했으며, 효과적인 마케팅 전략과 제품 현지화 덕분에 다음 분기에도 성장이 이어질 것으로 기대됩니다.",
    "reference": "회사는 2024년 3분기에 8%의 성장을 보고했으며, 이는 아시아 시장의 강력한 판매와 전략적 마케팅, 현지화된 제품 덕분으로 분석됩니다. 이러한 긍정적인 흐름은 다음 분기에도 지속될 것으로 예상됩니다."
}
single_test_data = SingleTurnSample(user_input=test_data["user_input"], response=test_data["response"], reference=test_data["reference"])

metric = BleuScore()
result = metric.single_turn_score(single_test_data)
print(f'bleu: {result:.4f}')

metric = RougeScore()
result = metric.single_turn_score(single_test_data)
print(f'rouge: {result:.4f}')

bleu: 0.3251
rouge: 1.0000


## RAG 품질 검사

Retrieval 단계 :
- ContextPrecision : (0,1) 주어진 Context들 중 실제 정답과 관련 있는가 --> 쓸데없이 많이 가져오지 않고 필요한 것만 가져왔는가?
- ContextRecall : (0,1) 정답(reference)에 포함된 핵심 내용이 얼마나 Context 안에 있는가 --> 빠뜨린 문서는 없는가?

Generation 단계 :
- Faithfulness : (0~1) 모델의 응답이 제공된 컨텍스트(context)에 얼마나 ‘충실하게’ 반영하는가 --> 환각이 있지는 않은가?
- ResponseRelevancy : (0~1) 모델의 응답(response)이 사용자의 질문(user_input)과 얼마나 관련이 있는지 --> 묻는 말에 답했는가?

핵심 : **정확히 가져오고, 제대로 참고해서, 적절히 답했는가**

RAG의 동작 원리에 따라 크게 3가지 요소로 나누어 평가할 수 있다.
1. Retrieval : 검색을 제대로 했는가?(contextRecall)
1. Grounded : 근거(context)를 기반으로 생성하였는가?(Faithfulness)
1. Relevancy : 사용자의 질문에 맞는 답변을 하였는가?(ResponseRelevancy)

https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/context_precision/

https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/context_recall/



In [9]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import LLMContextRecall

sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris.",
    reference="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["Paris is the capital of France."],
)

from ragas.llms import LangchainLLMWrapper
evaluator_llm = LangchainLLMWrapper(llm)
context_recall = LLMContextRecall(llm=evaluator_llm)
print('recall score:', await context_recall.single_turn_ascore(sample))

from ragas.metrics import LLMContextPrecisionWithoutReference

context_precision = LLMContextPrecisionWithoutReference(llm=evaluator_llm)

sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["The Brandenburg Gate is located in Berlin.", "The Eiffel Tower is located in Paris." ],
)
print('precision score:', await context_precision.single_turn_ascore(sample))

recall score: 0.0
precision score: 0.49999999995


In [10]:
from ragas.metrics import Faithfulness, ResponseRelevancy, ContextPrecision, ContextRecall

single = {
    "user_input": "회사의 경영 상황에 대한 기사 정보를 검색해줘.",
    "response": "회사는 2024년 3분기에 8%의 성장을 기록했으며, 효과적인 마케팅 전략과 제품 현지화 덕분에 다음 분기에도 성장이 이어질 것으로 기대됩니다.",
    "reference": "회사는 2024년 3분기에 8%의 성장을 보고했으며, 이는 아시아 시장의 강력한 판매와 전략적 마케팅, 현지화된 제품 덕분으로 분석됩니다. 이러한 긍정적인 흐름은 다음 분기에도 지속될 것으로 예상됩니다.",
    "retrieved_contexts": [
        "2023년 경제 악화로 기대 수익이 떨어졌다.",
        "경쟁사에게 특허를 소송당해 지금 법적 절차에 들어가고",
        "회사는 2024년 3분기에 8% 상승. 아시아 시장 실적 호조",
    ],
}

single_data = SingleTurnSample(
    user_input=single["user_input"],
    response=single["response"],
    reference=single["reference"],
    retrieved_contexts=single["retrieved_contexts"],
)

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(emb)

async def run_single_eval(sample):
    cp = await ContextPrecision(llm=evaluator_llm).single_turn_ascore(sample)
    cr = await ContextRecall(llm=evaluator_llm).single_turn_ascore(sample)
    f = await Faithfulness(llm=evaluator_llm).single_turn_ascore(sample)
    rr = await ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings).single_turn_ascore(sample)

    print(f"faithfulness: {f:.3f}")
    print(f"response_relevancy: {rr:.3f}")

await run_single_eval(single_data)

faithfulness: 0.250
response_relevancy: 0.303


In [11]:
rag_eval_dataset = [
    {
        "user_input": "슈퍼 요금제의 해지 위약금은 얼마인가요?",
        "response": "위약금은 최대 30,000원입니다.",
        "reference": "위약금은 잔여 약정기간에 비례하여 최대 30,000원이다.",
        "contexts": [
            "슈퍼 요금제는 데이터 10GB를 제공하며, 음성통화는 무제한이다.",
            "해지 위약금은 잔여 약정기간에 따라 부과되며 최대 30,000원이다.",
            "약정 기간 내 해지 시, 위약금이 발생합니다. 주의 바랍니다."
        ],
        "retrieved_contexts": [
            "프리미엄 요금제는 데이터 20GB 제공, 해지 위약금 최대 30,000원 부과됩니다.",
            "안심 요금제는 별도의 위약금이 없습니다.",
        ]
    },
    {
        "user_input": "환불 정책은 어떻게 되나요?",
        "response": "결제 후 전액 환불이 가능합니다.",
        "reference": "결제 후 7일 이내 미사용 시 전액 환불 가능. 7일 이상의 경우 일부 사용 이력이 있는 경우 환불 금액이 제한될 수 있습니다.",
        "contexts": [
            "결제 후 7일 이내에 서비스를 이용하지 않은 경우, 전액 환불이 가능합니다.",
            "일부 사용 이력이 있는 경우 환불 금액이 제한될 수 있습니다.",
        ],
        "retrieved_contexts": [
            "결제 후 7일 이내 미사용 시 전액 환불 가능.",
            "사용 이력이 있는 경우 환불이 제한됩니다.",
            "고객센터를 통해 환불 요청이 가능합니다.",
        ]
    },
    {
        "user_input": "내 요금제의 데이터 소진 후 속도는 어떻게 되나요?",
        "response": "데이터를 모두 사용하면 200Kbps로 속도가 제한됩니다. 비용을 추가로 지불하면 속도를 높일 수 있습니다.",
        "reference": "데이터 소진 후 속도는 400Kbps로 제한된다.",
        "contexts": [
            "일반 요금제는 데이터 5GB이 다 소진 되면 400Kbps로 제한된다.",
            "음성 통화는 기본 300분 제공됩니다.",
        ],
        "retrieved_contexts": [
            "기본 요금제의 데이터 제공량은 5GB이며, 소진 후 속도는 400Kbpsd입니다.",
            "데이터 무제한 요금제는 속도 제한이 없습니다. 마음놓고 사용하세요.",
        ]
    },
    {
        "user_input": "고객센터 운영시간은 언제인가요?",
        "response": "고객센터는 평일 9시부터 18시까지 운영됩니다.",
        "reference": "고객센터 운영시간은 평일 09:00~18:00입니다.",
        "contexts": [
            "고객센터는 평일 09:00~18:00 운영, \n\n  === 주말 및 공휴일 휴무 === ",
            "야간 문의는 챗봇을 사용하시면 친절하게 ai 상담사가 답해드립니다.",
        ],
        "retrieved_contexts": [
            "고객센터 운영시간: 평일 오전 9시부터 오후 6시까지.",
            "토요일은 근무하지 않습니다.",
        ]
    },
]


### evaluate를 사용해 한번에 평가하기

--> aevaluate를 사용하면 비동기 방식도 사용할 수 있습니다. https://docs.ragas.io/en/latest/references/aevaluate/

In [12]:
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_list(rag_eval_dataset)
print(evaluation_dataset)

azure_config = {
    "base_url": os.getenv('END_POINT'),
    "model_deployment": os.getenv('MODEL_NAME'),
    "model_name": os.getenv('MODEL_NAME'),
    "embedding_deployment": os.getenv('EMB_MODEL_NAME'),
    "embedding_name": os.getenv('EMB_MODEL_NAME'),
}

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(emb)

from ragas import evaluate
results = evaluate(evaluation_dataset, metrics=[Faithfulness(llm=evaluator_llm), ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings)])
df = results.to_pandas()
df

Evaluating: 100%|██████████| 8/8 [00:15<00:00,  1.97s/it]


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy
0,슈퍼 요금제의 해지 위약금은 얼마인가요?,"[프리미엄 요금제는 데이터 20GB 제공, 해지 위약금 최대 30,000원 부과됩니...","위약금은 최대 30,000원입니다.","위약금은 잔여 약정기간에 비례하여 최대 30,000원이다.",0.0,0.443110
1,환불 정책은 어떻게 되나요?,"[결제 후 7일 이내 미사용 시 전액 환불 가능., 사용 이력이 있는 경우 환불이 ...",결제 후 전액 환불이 가능합니다.,결제 후 7일 이내 미사용 시 전액 환불 가능. 7일 이상의 경우 일부 사용 이력이...,0.0,0.236709
2,내 요금제의 데이터 소진 후 속도는 어떻게 되나요?,"[기본 요금제의 데이터 제공량은 5GB이며, 소진 후 속도는 400Kbpsd입니다....",데이터를 모두 사용하면 200Kbps로 속도가 제한됩니다. 비용을 추가로 지불하면 ...,데이터 소진 후 속도는 400Kbps로 제한된다.,0.0,0.420755
3,고객센터 운영시간은 언제인가요?,"[고객센터 운영시간: 평일 오전 9시부터 오후 6시까지., 토요일은 근무하지 않습니다.]",고객센터는 평일 9시부터 18시까지 운영됩니다.,고객센터 운영시간은 평일 09:00~18:00입니다.,1.0,0.656736


## 직접 평가 항목을 설정하기

AspectCritic을 사용하면 llm을 사용해 평가 기준을 커스텀 할 수 있습니다.

말의 톤이라던지, 친절함 등도 평가가 가능합니다.

### 요약이 잘 되었는지 확인하기

In [13]:
from ragas import SingleTurnSample
from ragas.metrics import AspectCritic

test_data = {

    "user_input": "다음 글을 요약하세요.\n회사는 2024년 3분기에 8%의 상승을 기록했으며, 이는 아시아 시장에서의 강력한 실적에 힘입은 것입니다. 이 지역의 판매가 전체 성장에 크게 기여했습니다. 분석가들은 이러한 성공을 전략적인 마케팅과 제품 현지화 덕분으로 보고 있습니다. 아시아 시장의 긍정적인 추세는 다음 분기에도 이어질 것으로 예상됩니다.",
    "response": "회사는 2024년 3분기에 8%의 성장을 기록했으며, 효과적인 마케팅 전략과 제품 현지화 덕분에 다음 분기에도 성장이 이어질 것으로 기대됩니다.",
}

metric = AspectCritic(name="summary_accuracy",llm=evaluator_llm, definition="Verify if the summary is accurate.")
test_data = SingleTurnSample(user_input=test_data["user_input"], response=test_data["response"])
print(await metric.single_turn_ascore(test_data)) #결과가 1이면 pass, 0이면 fail

1


### 공손한 말투를 평가하는 예제

In [14]:
tone_eval_rows = [
    {
        "user_input": "배송 지연 이유를 알려주세요.",
        "response": "불편을 드려 정말 죄송합니다. 현재 물류센터 폭주로 평균 2~3일 지연되고 있어요. 원하시면 즉시 환불도 도와드릴게요.",
        "contexts": [],
        "retrieved_contexts": []
    },
    {
        "user_input": "교환 절차를 간단히 설명해 주세요.",
        "response": "교환은 마이페이지에서 신청하시면 됩니다. 추가로 궁금한 점 있으시면 도와드리겠습니다.",
        "contexts": [],
        "retrieved_contexts": []
    },
    {
        "user_input": "환불은 얼마나 걸리나요?",
        "response": "카드사마다 달라요.",
        "contexts": [],
        "retrieved_contexts": []
    },
    {
        "user_input": "비밀번호 재설정 방법 알려줘.",
        "response": "먼저 로그인 화면에서 '비밀번호 찾기'를 누르신 뒤 이메일 인증을 완료하시면 됩니다. 진행 중 어려움이 있으시면 제가 단계별로 안내해 드릴게요.",
        "contexts": [],
        "retrieved_contexts": []
    },
    {
        "user_input": "반말 쓰지 말아주세요.",
        "response": "알겠어. 그럼 그렇게 해 줄게.",
        "contexts": [],
        "retrieved_contexts": []
    },
]

In [15]:
from ragas import EvaluationDataset
ragas_dataset = EvaluationDataset.from_dict(tone_eval_rows)
print(ragas_dataset)

from ragas import evaluate
critic = """Evaluate whether the response is polite and respectful in Korean.
If it contains informal speech, imperative commands, or aggressive/rude wording, return 0; otherwise, return 1.
"""
metric = AspectCritic(name="tone",llm=evaluator_llm, definition=critic)
results = evaluate(ragas_dataset, metrics=[metric])
df = results.to_pandas()
print(df['tone'].aggregate('mean'))

df

Evaluating: 100%|██████████| 5/5 [00:04<00:00,  1.03it/s]


0.6


,user_input,retrieved_contexts,response,tone
0,배송 지연 이유를 알려주세요.,[],불편을 드려 정말 죄송합니다. 현재 물류센터 폭주로 평균 2~3일 지연되고 있어요....,1
1,교환 절차를 간단히 설명해 주세요.,[],교환은 마이페이지에서 신청하시면 됩니다. 추가로 궁금한 점 있으시면 도와드리겠습니다.,1
2,환불은 얼마나 걸리나요?,[],카드사마다 달라요.,0
3,비밀번호 재설정 방법 알려줘.,[],먼저 로그인 화면에서 '비밀번호 찾기'를 누르신 뒤 이메일 인증을 완료하시면 됩니다...,1
4,반말 쓰지 말아주세요.,[],알겠어. 그럼 그렇게 해 줄게.,0


## Multi-turn 대화 평가

In [16]:
from ragas.metrics import AspectCritic, SimpleCriteriaScore
from ragas.dataset_schema import MultiTurnSample, EvaluationDataset
from ragas.messages import HumanMessage, AIMessage
from ragas import evaluate

sample1 = MultiTurnSample(
    user_input=[
        HumanMessage(
            content="신용 한도를 올리고, 지난번 월마트 결제가 왜 거절됐는지 확인하고 싶어요."
        ),
        AIMessage(
            content="먼저 거절된 거래부터 확인해볼게요. 11월 20일 월마트에서 $234.56 결제가 잔액 부족으로 거절되었습니다."
        ),
        HumanMessage(content="그럴 리 없어요. 제 계좌에 돈이 충분했어요."),
        AIMessage(
            content="걱정되실 수 있겠네요. 계좌를 확인해보니, 현재 잔액은 $1,500이지만 당시 호텔 예약 보류금이 $800 잡혀 있어서 사용 가능한 잔액이 $700이었어요. 그로 인해 월마트 결제가 거절된 것으로 보입니다."
        ),
        HumanMessage(content="아, 맞아요. 호텔 예약을 깜빡했네요."),
        AIMessage(content="도움이 되었나요? 다른 도움을 드릴 게 있을까요?"),
        HumanMessage(content="그럼 신용 한도는 어떻게 올릴 수 있나요?"),
        AIMessage(
            content="현재 신용 점수가 740점이라 신용 한도 상향이 가능합니다. 최대 $2,000까지 늘릴 수 있지만, 이번 달에 세 건의 결제 거절 이력이 있어서 신중히 검토해보시길 권장드립니다."
        ),
    ]
)
sample2 = MultiTurnSample(
    user_input=[
        HumanMessage(
            content="최근 타깃 매장에서 결제가 거절된 이유를 알고 싶어요."
        ),
        AIMessage(
            content="거절된 거래를 확인해볼게요. 1월 5일 타깃 매장에서 $145.30 결제가 사기 의심 알림으로 인해 거절되었습니다."
        ),
        HumanMessage(content="이상하네요. 저는 아무 문제도 신고하지 않았어요."),
        AIMessage(
            content="이해합니다. 좀 더 자세히 살펴보니, 최근 다른 매장에서 높은 금액의 거래가 있어서 사기 방지 시스템이 이를 이상 거래로 판단하고, 그 영향으로 타깃 결제가 함께 거절된 것 같습니다."
        ),
        HumanMessage(content="아, 그렇군요. 그날 많이 쇼핑하긴 했어요."),
        AIMessage(content="도움이 되어 기쁩니다. 또 도와드릴 일이 있을까요?"),
    ]
)
sample3 = MultiTurnSample(
    user_input=[
        HumanMessage(
            content="안녕? 오늘 날씨가 춥네. 내일 기상 정보를 알려줘"
        ),
        AIMessage(
            content="날씨가 급격히 변하는 시기에는 감기에 걸리지 않게 따뜻한 유자차를 먹으면 좋아요"
        ),
        HumanMessage(content="날씨 알려달라니까?"),
        AIMessage(
            content="아 그렇군요. 제가 실수했습니다. 미국 날씨는 정보를 받을 수 없어 알려드릴 수 없습니다."
        ),
        HumanMessage(content="미국이 아니라 한국이야. 서울"),
        AIMessage(content="내일 서울의 날씨는 춥습니다."),
    ]
)
sample_list = [sample1, sample2, sample3]

In [17]:
definition = "Return 1 if the AI completes all Human requests fully without any rerequests; otherwise, return 0."
#"AI가 사용자의 모든 요청을 추가 요청 없이 완전히 수행한 경우 1을 반환하고, 그렇지 않으면 0을 반환한다."

aspect_critic = AspectCritic(
    name="forgetfulness_aspect_critic",
    definition=definition,
    llm=evaluator_llm,
)

In [18]:
result = evaluate(
    dataset=EvaluationDataset(samples=sample_list),
    metrics=[aspect_critic],
)

result.to_pandas()

Evaluating: 100%|██████████| 3/3 [00:03<00:00,  1.19s/it]


,user_input,forgetfulness_aspect_critic
0,"[{'content': '신용 한도를 올리고, 지난번 월마트 결제가 왜 거절됐는지 ...",1
1,"[{'content': '최근 타깃 매장에서 결제가 거절된 이유를 알고 싶어요.',...",1
2,"[{'content': '안녕? 오늘 날씨가 춥네. 내일 기상 정보를 알려줘', '...",0


In [19]:
from ragas.messages import HumanMessage,AIMessage,ToolMessage,ToolCall
from ragas.metrics import TopicAdherenceScore

multi_turn_sample = [
    HumanMessage(
        content="아인슈타인의 상대성 이론에 대해 자세히 설명해 줘"
    ),
    AIMessage(
        content="물론이죠. 관련된 정보를 찾아볼게요.",
        tool_calls=[
            ToolCall(
                name="document_search",
                args={"query": "아인슈타인의 상대성 이론"},
            )
        ],
    ),
    ToolMessage(
        content="관련 문서를 찾았습니다: 1. 『상대성 이론: 특수 및 일반 이론』, 2. 『일반 상대성 이론』 - A. 아인슈타인 저."
    ),
    AIMessage(
        content="아인슈타인의 상대성 이론에 관한 문서를 찾았습니다. '상대성 이론: 특수 및 일반 이론'과 '일반 상대성 이론(A. 아인슈타인 저)' 중 어느 쪽을 더 알고 싶으신가요?"
    ),
    HumanMessage(content="'일반 상대성 이론'에 대해 알려줘."),
    AIMessage(
        content="좋아요. 『일반 상대성 이론』에서 더 자세한 내용을 가져올게요.",
        tool_calls=[
            ToolCall(
                name="document_retrieve",
                args={"document": "일반 상대성 이론 (A. 아인슈타인 저)"},
            )
        ],
    ),
    ToolMessage(
        content="이 문서는 중력이 시공간의 구조에 어떤 영향을 미치는지를 설명하며, 질량과 시공간 곡률 사이의 관계를 다루고 있습니다."
    ),
    AIMessage(
        content="『일반 상대성 이론』은 중력이 시공간의 구조에 어떤 영향을 미치는지, 그리고 질량과 시공간 곡률이 어떤 관계에 있는지를 설명합니다. 더 자세한 설명이 필요하신가요, 아니면 이 정도로 충분하신가요?"
    ),
    HumanMessage(
        content="충분해. 그런데 혹시 초콜릿 케이크 맛있는 레시피도 알고 있어?"
    ),
    AIMessage(
        content="물론이죠! 간단하고 맛있는 초콜릿 케이크 레시피를 찾아볼게요.",
        tool_calls=[
            ToolCall(name="recipe_search", args={"query": "초콜릿 케이크 레시피"})
        ],
    ),
    ToolMessage(
        content="인기 있는 초콜릿 케이크 레시피를 찾았습니다. 재료: 밀가루, 설탕, 코코아가루, 달걀, 우유, 버터. 조리법: 재료를 섞고, 350°F(약 175°C)에서 30~35분간 구워주세요."
    ),
    AIMessage(
        content="좋은 초콜릿 케이크 레시피를 찾았어요! 전체 레시피를 자세히 알려드릴까요, 아니면 이 요약으로 괜찮으신가요?"
    ),
]

evaluator_llm = LangchainLLMWrapper(llm)
sample = MultiTurnSample(user_input=multi_turn_sample, reference_topics=["과학, 요리"])
scorer = TopicAdherenceScore(llm=evaluator_llm, mode="precision")
# score = await scorer.multi_turn_ascore(sample) # 비동기 방식
score = scorer.multi_turn_score(sample) # 동기 방식
print(score)

0.9999999999666667


## Agent 모델 평가

In [20]:
from ragas.metrics import ToolCallAccuracy

tool_call_sample = [
    HumanMessage(content="지금 뉴욕의 날씨가 어때?"),
    AIMessage(
        content="현재 뉴욕의 기온은 85°F이며, 부분적으로 흐립니다.",
        tool_calls=[ToolCall(name="weather_check", args={"location": "New York"})],
    ),
    HumanMessage(content="그걸 섭씨로 바꿔줘"),
    AIMessage(
        content="섭씨로 변환해드릴게요.",
        tool_calls=[
            ToolCall(
                name="temperature_conversion", args={"temperature_fahrenheit": 75}
            )
        ],
    ),
    ToolMessage(content="75°F는 약 23.9°C입니다."),
    AIMessage(content="75°F는 약 23.9°C입니다."),
]

sample = MultiTurnSample(
    user_input=tool_call_sample,
    reference_tool_calls=[
        ToolCall(name="weather_check", args={"location": "New York"}),
        ToolCall(
            name="temperature_conversion", args={"temperature_fahrenheit": 85}
        ),
    ],
)


scorer = ToolCallAccuracy()
score = await scorer.multi_turn_ascore(sample)
print(score)

0.5
